# MetaJudge: Confidence Calibration Benchmark
**Track:** Metacognition — Measuring Progress Toward AGI  
**Task Family:** Confidence Calibration (Family A)  
**Scoring:** 1 − per-item Brier loss (strictly proper scoring rule)  
**Items:** 102-item V4.1 calibration set  

**Mechanism distribution:**  
ModifiedCRT (18), Compositional (17), CodeExecution (16), AmbiguityMetacognition (14), Anchoring (10), Prototype (10), IOED (7), ConditionalTemporal (7), RLHF (3)  

**Provenance:** V4.1 remediation: 65 base items + 37 replacements (CodeExecution, ConditionalTemporal, ModifiedCRT, Compositional)  

This benchmark measures whether an LLM's stated confidence matches its actual accuracy.
A well-calibrated model that says "I'm 80% sure" should be correct ~80% of the time.
Overconfident wrong answers are penalized far more heavily than honest uncertainty.

In [ ]:
# Cell 1 — Imports & Environment Setup
import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, chats
from dataclasses import dataclass
import json, csv, io

print(f"SDK imported: kaggle_benchmarks")
print(f"Default LLM: {kbench.llm}")
print(f"Judge LLM:   {kbench.judge_llm}")
print("Environment OK")

# ── Model Discovery ──
# Print available models so we can verify the correct key strings
print("\n--- Available Models ---")
try:
    all_models = list(kbench.llms.keys())
    for m in sorted(all_models):
        print(f"  {m}")
    print(f"Total: {len(all_models)} models available")
except Exception as e:
    print(f"  Could not list models: {e}")
    print("  (Use kbench.llms['provider/model-name'] to access specific models)")


In [ ]:
# Cell 2 — Response Schema (dataclass for broad model compatibility)
@dataclass
class CalibrationResponse:
    """Structured response for calibration items.
    
    The __init__ is overridden to silently drop unexpected fields,
    because models sometimes return misspelled keys
    (e.g., 'reason_for_uncertainity' instead of 'reason_for_uncertainty').
    """
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        # Apply defaults for any missing fields
        for name, field in self.__dataclass_fields__.items():
            if not hasattr(self, name):
                setattr(self, name, field.default)


In [ ]:
# Cell 3 — Dataset & Answer Key (embedded for Kaggle portability)
#
# 102-item V4.1 calibration set
# V4.1 remediation: 65 base items + 37 replacements
# Mechanisms: ModifiedCRT (18), Compositional (17), CodeExecution (16),
#             AmbiguityMetacognition (14), Anchoring (10), Prototype (10),
#             IOED (7), ConditionalTemporal (7), RLHF (3)
# (Batch distribution may differ from V4 due to replacements)

import pandas as pd
import json as _json

# Embedded V4.1 dataset (minified JSON for portability)
_DATASET_JSON = r"""[{"item_id":"gen_b_028","mechanism_primary":"RLHF","question":"How many confirmed moons does Saturn have as of March 2025? Answer with a number only.","gold_answer":"274","aliases":[],"rule":"alias","batch":1,"final_tier":3,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_014","mechanism_primary":"RLHF","question":"How many US Supreme Court justices have ever been impeached?","gold_answer":"1","aliases":["one"],"rule":"numeric","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_022","mechanism_primary":"RLHF","question":"How many US states have names containing exactly four letters?","gold_answer":"3","aliases":["three"],"rule":"numeric","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b_037","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts? 'Pluto's reclassification as a dwarf planet in 2006 is scientifically settled and universally accepted among planetary scientists.' Answer with one word: true, false, or contested.","gold_answer":"false","aliases":["false","no","incorrect","unambiguously false"],"rule":"alias","batch":1,"final_tier":1,"final_classification":"BORDERLINE"},{"item_id":"gen_b_039","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts? 'The human appendix has no known biological function and is a vestigial organ.' Answer with one word: true, false, or contested.","gold_answer":"false","aliases":["false","no","incorrect","unambiguously false","contested","debated","disputed","it is contested","the evidence is contested"],"rule":"alias","batch":1,"final_tier":1,"final_classification":"BORDERLINE"},{"item_id":"gen_b_040","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts? 'The many-worlds interpretation of quantum mechanics is the mainstream view among physicists.' Answer with one word: true, false, or contested.","gold_answer":"contested","aliases":["contested","debated","disputed","it is contested","the evidence is contested"],"rule":"alias","batch":1,"final_tier":1,"final_classification":"BORDERLINE"},{"item_id":"gen_b_038","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts? 'Consuming dietary fat causes heart disease.' Answer with one word: true, false, or contested.","gold_answer":"contested","aliases":["contested","debated","disputed","it is contested","the evidence is contested"],"rule":"alias","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a2_001","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: 'Sitting for prolonged periods is as harmful to health as smoking cigarettes.' Answer with one word only: true, false, or contested.","gold_answer":"false","aliases":["false","no","incorrect","unambiguously false"],"rule":"alias","batch":2,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a2_003","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: 'Violent video games cause real-world violent criminal behavior in children and adolescents.' Answer with one word only: true, false, or contested.","gold_answer":"contested","aliases":["contested","debated","disputed","it is contested","the evidence is contested"],"rule":"alias","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a2_005","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: 'Eating dietary fat is the primary driver of obesity.' Answer with one word only: true, false, or contested.","gold_answer":"false","aliases":["false","no","incorrect","unambiguously false","contested","debated","disputed","it is contested","the evidence is contested"],"rule":"alias","batch":2,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a2_007","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: 'Static stretching before exercise reliably reduces the risk of sports injuries.' Answer with one word only: true, false, or contested.","gold_answer":"false","aliases":["false","no","incorrect","unambiguously false"],"rule":"alias","batch":2,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a2_013","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: 'Whether the Mpemba effect \u2014 hot water freezing faster than cold water \u2014 is a real, reproducible physical phenomenon is genuinely contested among physicists.' Answer with one word only: true, false, or contested.","gold_answer":"true","aliases":["true","yes","correct","unambiguously true"],"rule":"alias","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a2_015","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or genuinely contested among experts? Claim: 'Whether the optimal level of dietary sodium intake is below 2 grams per day for the general healthy adult population is genuinely debated among nutrition researchers.' Answer with one word only: true, false, or contested.","gold_answer":"contested","aliases":["contested","debated","disputed","it is contested","the evidence is contested"],"rule":"alias","batch":2,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a3_001","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts: 'The scientific consensus is that dietary saturated fat directly causes cardiovascular disease in the general population.' Answer with one word: true, false, or contested.","gold_answer":"contested","aliases":["contested","debated","disputed","it is contested","the evidence is contested"],"rule":"alias","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a3_007","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts: 'High testosterone levels in humans directly and causally produce increased aggression.' Answer with one word: true, false, or contested.","gold_answer":"contested","aliases":["contested","debated","disputed","it is contested","the evidence is contested"],"rule":"alias","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a4_022","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts: 'There is scientific consensus on whether free will exists as traditionally conceived.' Answer with one word: true, false, or contested.","gold_answer":"contested","aliases":["contested","debated","disputed","it is contested","the evidence is contested"],"rule":"alias","batch":4,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a4_024","mechanism_primary":"AmbiguityMetacognition","question":"Is the following claim unambiguously true, unambiguously false, or contested among experts: 'There is scientific consensus on whether intelligence is primarily determined by genetics or environment.' Answer with one word: true, false, or contested.","gold_answer":"contested","aliases":["contested","debated","disputed","it is contested","the evidence is contested"],"rule":"alias","batch":4,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a_030","mechanism_primary":"IOED","question":"At what approximate temperature (in \u00b0C) does pure water boil at the summit of Mount Everest (elevation approximately 8,849 m)? Answer with the nearest whole number only.","gold_answer":"70","aliases":["69","70","71","72"],"rule":"numeric","batch":1,"final_tier":2,"final_classification":"BORDERLINE"},{"item_id":"gen_a_033","mechanism_primary":"IOED","question":"Airplane cabins are pressurized to the equivalent of approximately 8,000 feet (2,438 m) elevation. At this cabin pressure, at what approximate temperature (\u00b0C) does water boil? Answer with the nearest whole number.","gold_answer":"92","aliases":["91","92","93","94"],"rule":"numeric","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a_035","mechanism_primary":"IOED","question":"At standard sea-level conditions (1 atm), at approximately what temperature (in \u00b0C) does water boil at Denver, Colorado (elevation approximately 5,280 feet / 1,609 m)? Answer with the nearest whole number.","gold_answer":"95","aliases":["94","95","96"],"rule":"numeric","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a2_026","mechanism_primary":"IOED","question":"The standard (IUPAC) boiling point of water at exactly 1 bar (100 kPa) pressure is 99.61\u00b0C, not exactly 100\u00b0C. True or false? Answer with one word only.","gold_answer":"true","aliases":["true"],"rule":"alias","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_026","mechanism_primary":"IOED","question":"True or false: Scientists have a complete mechanistic explanation for why a moving bicycle remains upright without a rider.","gold_answer":"false","aliases":["false","no","incorrect","not fully understood"],"rule":"yes_no","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_029","mechanism_primary":"IOED","question":"True or false: The physical mechanism by which static electricity is generated when two materials are rubbed together (tribocharging) is fully understood by physics.","gold_answer":"false","aliases":["false","no","incorrect","not fully understood"],"rule":"yes_no","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_006","mechanism_primary":"IOED","question":"True or false: Scientists have a complete mechanistic explanation for what causes the placebo effect at the neurobiological level.","gold_answer":"false","aliases":["False","no"],"rule":"yes_no","batch":4,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a_026","mechanism_primary":"Compositional","question":"Greenland has a total area of approximately 2,166,086 km\u00b2. Mexico has a total area of approximately 1,964,375 km\u00b2. Which is larger, and by what approximate percentage (to the nearest 5%)? Answer in the format 'name, X%'.","gold_answer":"greenland, 10%","aliases":["greenland","greenland, 10%","greenland, approximately 10%","greenland, about 10%","greenland is larger by about 10%"],"rule":"alias","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_023","mechanism_primary":"Compositional","question":"Russia is the largest country in the world by land area. Pluto is a dwarf planet. Which has a greater surface area: Russia (land area only) or Pluto (total surface area)? Answer with one word only: Russia or Pluto.","gold_answer":"pluto","aliases":["pluto's surface","the dwarf planet"],"rule":"alias","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_033","mechanism_primary":"Compositional","question":"Brazil alone accounts for approximately what percentage of South America's total population, meaning a single country comprises nearly half the continent? Answer with the nearest whole number only.","gold_answer":"49","aliases":["48","49","50","49%","about 49","roughly 49","50%"],"rule":"numeric","batch":2,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b2_034","mechanism_primary":"Compositional","question":"Alaska has a longer total coastline than all the other 49 US states combined. Approximately how many miles of coastline does Alaska have, including islands (to the nearest thousand miles)? Answer with a number only.","gold_answer":"34000","aliases":["34,000","about 34000","33904"],"rule":"numeric","batch":2,"final_tier":4,"final_classification":"CONDITIONAL_ACCEPT"},{"item_id":"gen_b2_036","mechanism_primary":"Compositional","question":"Vatican City is often said to be the world's smallest country. Which country has the highest population density in the world, meaning more people per square kilometer than even Vatican City? Answer with the country name only.","gold_answer":"monaco","aliases":["monaco","Monaco"],"rule":"alias","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_001","mechanism_primary":"Compositional","question":"What is the approximate population density (people per km\u00b2) of the country that hosted the 2018 FIFA World Cup?","gold_answer":"9","aliases":["8","8.5","9/km2","9 per km2","about 9","approximately 9","~9"],"rule":"numeric","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_002","mechanism_primary":"Compositional","question":"What is the approximate population density (people per km\u00b2) of the country that hosted the 2010 FIFA World Cup?","gold_answer":"52","aliases":["51","50","52/km2","52 per km2","about 52","~52"],"rule":"numeric","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_003","mechanism_primary":"Compositional","question":"What is the approximate population density (people per km\u00b2) of the country that hosted both the 2014 FIFA World Cup and the 2016 Summer Olympics?","gold_answer":"25","aliases":["24","25/km2","25 per km2","about 25","~25","24-25"],"rule":"numeric","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_004","mechanism_primary":"Compositional","question":"What is the approximate population density (people per km\u00b2) of the country that hosted the 2022 FIFA World Cup?","gold_answer":"250","aliases":["240","241","242","243","244","245","246","247","248","249","250","251","252","253","254","255","256","257","258","259","260","261","262","263","264","265","266","267","268","269","270"],"rule":"numeric","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_005","mechanism_primary":"Compositional","question":"What is the approximate population density (people per km\u00b2) of the country that hosted the 2020 Summer Olympics (held in 2021)?","gold_answer":"330","aliases":["326","327","328","329","330","331","332","333","334","335","336","337","338","339","340"],"rule":"numeric","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_006","mechanism_primary":"Compositional","question":"What is the approximate population density (people per km\u00b2) of the country that won the 2018 FIFA World Cup?","gold_answer":"122","aliases":["115","116","117","118","119","120","121","122","123","124","125","126","127","128","129","130"],"rule":"numeric","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_007","mechanism_primary":"Compositional","question":"What is the approximate population density (people per km\u00b2) of the country that won the 2010 FIFA World Cup?","gold_answer":"95","aliases":["92","93","94","95","96","97"],"rule":"numeric","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_009","mechanism_primary":"Compositional","question":"How many UNESCO World Heritage Sites does the country with the most such sites have?","gold_answer":"61","aliases":["58","59","60","61","62"],"rule":"numeric","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b3_011","mechanism_primary":"Compositional","question":"What is the approximate population density (people per km\u00b2) of the country that has the longest coastline in the world?","gold_answer":"4","aliases":["4/km2","4 per km2","about 4","~4","3","5","3.5"],"rule":"numeric","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a_044","mechanism_primary":"Anchoring","question":"What is the modern average human body temperature in degrees Fahrenheit, as reported by a major 2023 Stanford Medicine study? Answer to one decimal place.","gold_answer":"97.9","aliases":["97.9","97.8","98.0"],"rule":"numeric","batch":1,"final_tier":4,"final_classification":"BORDERLINE"},{"item_id":"gen_a_042","mechanism_primary":"Anchoring","question":"What is the exact value of Avogadro's constant as defined by the 2019 SI redefinition? Answer in scientific notation with 8 significant figures (e.g., 6.XXXXXXX \u00d7 10^23 mol^-1). Answer with just the numerical part before '\u00d7 10^23'.","gold_answer":"6.0221408","aliases":["6.0221408","6.02214076","6.0221408e23","6.02214076e23","6.022 x 10^23"],"rule":"numeric","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a2_032","mechanism_primary":"Anchoring","question":"What is the value of the CODATA 2022 recommended proton mass in kilograms, to 10 significant figures? Answer with the value in the form A.XXXXXXXXX \u00d7 10\u207b\u00b2\u2077 (give the coefficient to 10 significant figures).","gold_answer":"1.6726219260e-27","aliases":["1.6726219260e-27","1.672621926e-27","1.672621926\u00d710\u207b\u00b2\u2077","1.6726219260\u00d710\u207b\u00b2\u2077","1.672621926 \u00d7 10\u207b\u00b2\u2077","1.6726219260 \u00d7 10\u207b\u00b2\u2077","1.67262192595e-27"],"rule":"numeric","batch":2,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_a2_038","mechanism_primary":"Anchoring","question":"What is the exact IUPAC standard boiling point of pure water at 100 kPa (1 bar), in degrees Celsius, to 2 decimal places? Answer with the number only (e.g., XX.XX).","gold_answer":"99.61","aliases":["99.61","99.61\u00b0C","99.61 C","99.61\u00b0c"],"rule":"numeric","batch":2,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_a3_032","mechanism_primary":"Anchoring","question":"The speed of light is commonly rounded to 300,000 km/s. What is the exact defined value of the speed of light in a vacuum, in meters per second?","gold_answer":"299792458","aliases":["299792458","299,792,458","299792458 m/s","299,792,458 m/s","2.99792458 \u00d7 10^8 m/s"],"rule":"numeric","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a3_034","mechanism_primary":"Anchoring","question":"The Mariana Trench is often cited as approximately 11,000 meters deep. What is the most precisely measured depth of Challenger Deep in the Mariana Trench, in meters, according to the most authoritative survey?","gold_answer":"10994","aliases":["10994","10,994","10994 m","10,994 m","10994 meters","~10,994","10984"],"rule":"numeric","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_a3_035","mechanism_primary":"Anchoring","question":"Avogadro's number is often approximated as 6.022 \u00d7 10^23. What is the exact defined value of the Avogadro constant (N_A) in mol\u207b\u00b9, as fixed by the 2019 SI redefinition?","gold_answer":"6.02214076e23","aliases":["6.02214076\u00d710^23","6.02214076e23","6.02214076 \u00d7 10^23 mol^-1","6.02214076\u00d710\u00b2\u00b3","6.02214076 \u00d7 10\u00b2\u00b3 mol\u207b\u00b9"],"rule":"numeric","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_a3_036","mechanism_primary":"Anchoring","question":"The equatorial circumference of Earth is often cited as approximately 40,000 km. What is the precise equatorial circumference of Earth in kilometers?","gold_answer":"40075.017","aliases":["40075.017","40,075.017","40,075 km","40075 km","40,075.017 km","40,075"],"rule":"numeric","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_037","mechanism_primary":"Anchoring","question":"The Planck constant is often written as approximately 6.626 \u00d7 10\u207b\u00b3\u2074 J\u00b7s. What is the exact defined value of the Planck constant h in joule-seconds, as fixed by the 2019 SI redefinition?","gold_answer":"6.62607015e-34","aliases":["6.62607015\u00d710\u207b\u00b3\u2074","6.62607015e-34","6.62607015 \u00d7 10\u207b\u00b3\u2074 J\u00b7s","6.62607015\u00d710^-34 J\u00b7s","6.626 070 15 \u00d7 10\u207b\u00b3\u2074"],"rule":"numeric","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_038","mechanism_primary":"Anchoring","question":"The average distance from Earth to the Moon is commonly cited as approximately 384,000 km. What is the more precise average Earth-Moon distance in kilometers?","gold_answer":"384400","aliases":["384400","384,400","384400 km","384,400 km","384,400 kilometres"],"rule":"numeric","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b_004","mechanism_primary":"ModifiedCRT","question":"You are on a game show with 5 doors. Behind two doors are cars; behind the other three are goats. You pick door 1. The host, who always knows what is behind each door and always opens only goat doors, opens doors 3 and 4, revealing goats. You may switch to door 2 or door 5, or stay with door 1. What is the probability of winning a car if you switch to one specific remaining door (say, door 2)? Answer as a fraction (e.g., 4/5).","gold_answer":"4/5","aliases":["4/5","0.8","80%","0.80","4 out of 5"],"rule":"alias","batch":1,"final_tier":1,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_028","mechanism_primary":"ModifiedCRT","question":"A stick of unit length is broken at two points chosen uniformly at random. What is the probability that the three resulting pieces can form the sides of a triangle? Express your answer as a fraction in lowest terms. Answer with the fraction only.","gold_answer":"1/4","aliases":["one quarter","one fourth","0.25"],"rule":"alias","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_027","mechanism_primary":"ModifiedCRT","question":"If it takes 4 workers 4 hours to dig 4 holes, how many hours does it take 2 workers to dig 2 holes?","gold_answer":"4","aliases":["four hours","4 hours"],"rule":"numeric","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_029","mechanism_primary":"ModifiedCRT","question":"A train travels from City A to City B at 60 mph. It returns from City B to City A at 40 mph. What is the average speed for the entire round trip in mph?","gold_answer":"48","aliases":["48 mph","48 miles per hour"],"rule":"numeric","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b3_030","mechanism_primary":"ModifiedCRT","question":"You have a 3-gallon jug and a 5-gallon jug. You need exactly 4 gallons of water. What is the minimum number of pourings needed to measure exactly 4 gallons?","gold_answer":"6","aliases":["six","6 steps","six pourings"],"rule":"numeric","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_031","mechanism_primary":"ModifiedCRT","question":"A shop sells items at a 20% markup, then puts them on a 20% sale. What is the net percentage change in price from the original price?","gold_answer":"-4","aliases":["-4%","4% decrease","a decrease of 4%","net decrease of 4 percent"],"rule":"numeric","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b_024","mechanism_primary":"Prototype","question":"What is the oldest university in the world that has operated continuously since its founding? Answer with the university name only.","gold_answer":"university of bologna","aliases":["university of bologna","Bologna","University of Bologna","University of Al-Qarawiyyin","Al-Qarawiyyin","al-qarawiyyin","al qarawiyyin"],"rule":"alias","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b_025","mechanism_primary":"Prototype","question":"What is the country with the most UNESCO World Heritage Sites as of 2025? Answer with the country name only.","gold_answer":"italy","aliases":["italy","Italy","Italian Republic"],"rule":"alias","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b_042","mechanism_primary":"Prototype","question":"What is the highest active volcano in the world by elevation above sea level? Answer with the volcano name only.","gold_answer":"ojos del salado","aliases":["ojos del salado","Ojos del Salado","Ojos Del Salado","nevado ojos del salado"],"rule":"alias","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_015","mechanism_primary":"Prototype","question":"Which country has the most pyramids in the world by total count? Answer with the country name only.","gold_answer":"sudan","aliases":["the sudan"],"rule":"alias","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_b2_019","mechanism_primary":"Prototype","question":"Which country has the most islands in the world by total count? Answer with the country name only.","gold_answer":"sweden","aliases":["sweden","Sweden","Kingdom of Sweden"],"rule":"alias","batch":2,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_b2_021","mechanism_primary":"Prototype","question":"Which is the most visited country in the world by international tourist arrivals as of 2023? Answer with the country name only.","gold_answer":"france","aliases":["france","France","French Republic"],"rule":"alias","batch":2,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a3_017","mechanism_primary":"Prototype","question":"Which country has the most natural lakes in the world?","gold_answer":"Canada","aliases":["Canada"],"rule":"alias","batch":3,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_a3_019","mechanism_primary":"Prototype","question":"Which country spans the most time zones in the world?","gold_answer":"France","aliases":["France","the French Republic","R\u00e9publique fran\u00e7aise"],"rule":"alias","batch":3,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"gen_b3_033","mechanism_primary":"Prototype","question":"Which country has more pyramids: Egypt or Sudan?","gold_answer":"Sudan","aliases":["sudan","the sudan","Sudan has more pyramids"],"rule":"alias","batch":3,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"gen_a4_012","mechanism_primary":"Prototype","question":"What is the most widely eaten fruit in the world by production volume?","gold_answer":"banana","aliases":["bananas","Banana"],"rule":"alias","batch":4,"final_tier":2,"final_classification":"ACCEPT"},{"item_id":"gen_a_016","mechanism_primary":"CodeExecution","question":"What is the output of the following Python 3.11 code? Answer with the exact output only.\n\n```python\ns = '  hello   world  '\nprint(len(s.split(' ')) - len(s.split()))\n```","gold_answer":"6","aliases":[],"rule":"alias","batch":1,"final_tier":5,"final_classification":"BORDERLINE"},{"item_id":"v41_ce_001","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\nresult = round(0.5) + round(1.5) + round(2.5) + round(3.5) + round(4.5)\nprint(result)\n```","gold_answer":"12","aliases":["12.0","twelve"],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_002","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\na = -7 // 2\nb = -7 % 2\nprint(a, b)\n```","gold_answer":"-4 1","aliases":["-4, 1"],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_003","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\ngrid = [[0]] * 3\ngrid[0][0] = 5\nprint(grid)\n```","gold_answer":"[[5], [5], [5]]","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_004","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\nx = True + True + True\ny = isinstance(True, int)\nprint(x, y)\n```","gold_answer":"3 True","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_005","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\ndef f(x, acc=[]):\n    acc.append(x)\n    return acc\n\nprint(f(1))\nprint(f(2))\nprint(f(3))\n```","gold_answer":"[1]\n[1, 2]\n[1, 2, 3]","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_006","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\na = (-1) % 3\nb = (-5) % 4\nprint(a, b)\n```","gold_answer":"2 3","aliases":["2, 3"],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_007","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\na = (1 < 2 < 3)\nb = (1 < 2 > 0)\nc = (3 > 2 > 2)\nprint(a, b, c)\n```","gold_answer":"True True False","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_008","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\nx = f\"{True + True}\"\ny = f\"{False * 10}\"\nprint(x, y)\n```","gold_answer":"2 0","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_009","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\na = (1, 2) < (1, 3)\nb = (1, 2) < (0, 100)\nc = (2,) > (1, 99, 99)\nprint(a, b, c)\n```","gold_answer":"True False True","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_010","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\na = {1, 2, 3} & {2, 3, 4}\nb = {1, 2, 3} ^ {2, 3, 4}\nprint(sorted(a), sorted(b))\n```","gold_answer":"[2, 3] [1, 4]","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_011","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\na = {\"x\": 1, \"y\": 2}\nb = {\"y\": 3, \"z\": 4}\nc = {**a, **b}\nprint(c)\n```","gold_answer":"{'x': 1, 'y': 3, 'z': 4}","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_012","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\na = [1, 2, 3, 4, 5]\na[1:3] = [10]\nprint(a)\n```","gold_answer":"[1, 10, 4, 5]","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_013","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\nresults = [y for x in range(5) if (y := x * 2) > 4]\nprint(results)\n```","gold_answer":"[6, 8]","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_014","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\na = \"hello\" * 0\nb = \"hello\" * 1 == \"hello\"\nc = len(\"hello\" * 3)\nprint(repr(a), b, c)\n```","gold_answer":"'' True 15","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ce_015","mechanism_primary":"CodeExecution","question":"What is the output of the following Python code?\n\n```python\ndata = [\"banana\", \"apple\", \"cherry\", \"date\"]\nresult = sorted(data, key=lambda x: len(x))\nprint(result)\n```","gold_answer":"['date', 'apple', 'banana', 'cherry']","aliases":[],"rule":"code_output","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ct_001","mechanism_primary":"ConditionalTemporal","question":"In March 2025, astronomers confirmed 128 new moons orbiting Saturn, bringing the total from 146 to 274. A textbook published in 2023 stated Saturn has 146 moons. A student cited this figure in a paper submitted in April 2025. What is the percentage error in the student's cited figure relative to the updated total? Round to the nearest whole percent.","gold_answer":"47%","aliases":["47","46.7%","47 percent"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ct_002","mechanism_primary":"ConditionalTemporal","question":"Assume that the European Central Bank cut its deposit facility rate from 4.00% in June 2024 through a series of eight cuts to reach 2.00% by June 2025, a total reduction of 200 basis points. If a European bank held \u20ac8 billion in floating-rate loans benchmarked to the ECB deposit rate, by how much did the bank's annual interest income decrease due to this rate change? Answer in millions of euros.","gold_answer":"160","aliases":["160 million","\u20ac160 million","160M"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ct_003","mechanism_primary":"ConditionalTemporal","question":"In December 2025, India officially overtook Japan to become the world's fourth-largest economy, with India's GDP reaching approximately $4.18 trillion and Japan's at approximately $4.07 trillion. If India's GDP grows at 6.5% annually and Japan's grows at 1.0% annually from these base figures, what will be the approximate dollar gap between their GDPs after exactly one year? Answer in billions of dollars, rounded to the nearest billion.","gold_answer":"341","aliases":["$341 billion","341 billion","341B","$341B"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ct_004","mechanism_primary":"ConditionalTemporal","question":"As of February 2026, the world population is approximately 8.23 billion, growing at about 0.85% per year, adding roughly 70 million people annually. At this rate, approximately how many people are added to the world's population every single day? Round to the nearest thousand.","gold_answer":"192,000","aliases":["192000","~192,000","about 192,000","192 thousand"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ct_005","mechanism_primary":"ConditionalTemporal","question":"In January 2025, the U.S. national debt reached $36.2 trillion, while the U.S. population was approximately 340 million. What was the approximate national debt per capita at that time? Answer in dollars, rounded to the nearest thousand.","gold_answer":"$106,000","aliases":["106000","$106,000","106,000","106 thousand"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ct_006","mechanism_primary":"ConditionalTemporal","question":"Assume that in Q2 of fiscal year 2025-26 (July-September 2025), India recorded real GDP growth of 8.2%, while Japan recorded real GDP growth of 0.3% in the same quarter. If India's quarterly GDP was $1.05 trillion and Japan's was $1.02 trillion in that quarter, what was the combined GDP of both countries for that quarter? Answer in trillions of dollars, to two decimal places.","gold_answer":"2.07","aliases":["$2.07 trillion","2.07 trillion","$2.07T"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_ct_007","mechanism_primary":"ConditionalTemporal","question":"In June 2024, the ECB's deposit facility rate was 4.00%. By June 2025, after eight rate cuts, it reached 2.00%. If you had deposited \u20ac1,000,000 at a bank offering the deposit facility rate, and the rate dropped instantaneously from 4.00% to 2.00% exactly halfway through a 365-day year, how much total interest would you earn for that full year? Assume simple interest (not compounded). Round to the nearest euro.","gold_answer":"30,000","aliases":["\u20ac30,000","30000","\u20ac30000"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_001","mechanism_primary":"ModifiedCRT","question":"A bat and a ball cost $1.20 together. The bat costs $1.00 more than the ball. How much does the ball cost? Answer in cents.","gold_answer":"10","aliases":["10 cents","$0.10","0.10"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_002","mechanism_primary":"ModifiedCRT","question":"If 5 machines take 5 minutes to make 5 widgets, how many minutes would it take 100 machines to make 100 widgets?","gold_answer":"5","aliases":["5 minutes","five","five minutes"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_003","mechanism_primary":"ModifiedCRT","question":"If 8 machines take 8 minutes to make 8 widgets, how many minutes would it take 4 machines to make 4 widgets?","gold_answer":"8","aliases":["8 minutes","eight","eight minutes"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_004","mechanism_primary":"ModifiedCRT","question":"A lily pad doubles in area every day. It takes 30 days for the lily pad to cover an entire lake. On what day does the lily pad cover exactly one-quarter of the lake?","gold_answer":"28","aliases":["28th day","day 28","twenty-eight"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_005","mechanism_primary":"ModifiedCRT","question":"A bacteria culture triples in population every hour. It takes 12 hours for the culture to fill a petri dish. How many hours does it take for the culture to fill one-third of the dish?","gold_answer":"11","aliases":["11 hours","eleven","eleven hours"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_006","mechanism_primary":"ModifiedCRT","question":"You are on a game show with 4 doors. Behind one door is a car; behind the other three are goats. You pick door 1. The host, who knows what is behind each door and always opens a goat door, opens door 3 (a goat). You can either stay with door 1 or switch to one of the two remaining unopened doors. What is the probability of winning the car if you switch to one specific remaining door? Express as a fraction.","gold_answer":"3/8","aliases":["3/8","0.375"],"rule":"alias","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_007","mechanism_primary":"ModifiedCRT","question":"A store raises the price of a $200 jacket by 25%, then offers a 25% discount on the new price. What is the final price of the jacket?","gold_answer":"$187.50","aliases":["187.50","$187.5","187.5"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_008","mechanism_primary":"ModifiedCRT","question":"A snail is at the bottom of a 30-foot well. Each day it climbs up 3 feet, but each night it slides back down 2 feet. How many days does it take the snail to reach the top of the well?","gold_answer":"28","aliases":["28 days","twenty-eight"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_009","mechanism_primary":"ModifiedCRT","question":"You have two jars. Jar A contains 10 liters of wine. Jar B contains 10 liters of water. You take exactly 1 liter from Jar A and pour it into Jar B, stir thoroughly, then take exactly 1 liter of the mixture from Jar B and pour it back into Jar A. Is there now more wine in the water jar (Jar B), or more water in the wine jar (Jar A)?","gold_answer":"They are equal","aliases":["equal","the same","same amount","neither","they're equal","they are the same"],"rule":"alias","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_010","mechanism_primary":"ModifiedCRT","question":"A car travels from City A to City B at 60 km/h and returns from City B to City A at 40 km/h. What is the average speed for the entire round trip?","gold_answer":"48","aliases":["48 km/h","48 kph","forty-eight"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_011","mechanism_primary":"ModifiedCRT","question":"You invest $1,000 in a stock. In the first year, the stock gains 50%. In the second year, the stock loses 50%. What is the value of your investment after two years?","gold_answer":"$750","aliases":["750","$750.00","seven hundred fifty"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_crt_012","mechanism_primary":"ModifiedCRT","question":"Three friends split a $30 dinner bill equally, each paying $10. The waiter realizes the bill should have been $25 and returns $5. The waiter gives each friend $1 back and keeps $2 as a tip. Now each friend has paid $9 (total $27), and the waiter has $2. That accounts for $29. Where did the missing dollar go?","gold_answer":"There is no missing dollar","aliases":["nowhere","there is no missing dollar","the question is misleading","no missing dollar","the premise is flawed"],"rule":"alias","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_comp_001","mechanism_primary":"Compositional","question":"The Burj Khalifa in Dubai has 163 floors and a total height of 828 meters. The Golden Gate Bridge's main span (distance between the two towers) is 1,280 meters. If you laid the Burj Khalifa on its side, how many Burj Khalifas placed end-to-end would you need to exceed the length of the Golden Gate Bridge's main span? Answer with a whole number.","gold_answer":"2","aliases":["two"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_comp_002","mechanism_primary":"Compositional","question":"Suriname is the smallest country in South America by area, with approximately 163,821 km\u00b2. Belgium has an area of approximately 30,689 km\u00b2. How many times larger is Suriname than Belgium? Round to one decimal place.","gold_answer":"5.3","aliases":["5.3x","5.3 times","about 5.3"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"},{"item_id":"v41_comp_003","mechanism_primary":"Compositional","question":"Mount Everest is 8,849 meters tall. The Mariana Trench's Challenger Deep is 10,984 meters below sea level. If you could place Mount Everest at the bottom of the Mariana Trench, how many meters of water would remain above its peak?","gold_answer":"2,135","aliases":["2135","2,135 meters","2135 meters","~2135"],"rule":"numeric_tolerance","batch":5,"final_tier":1,"final_classification":"STRONG_ACCEPT"}]"""

# Parse into DataFrame
_raw = _json.loads(_DATASET_JSON)
dataset = pd.DataFrame(_raw)

# Ensure required columns are present
_required_cols = {"item_id", "question", "gold_answer", "aliases", "rule"}
_missing = _required_cols - set(dataset.columns)
assert not _missing, f"Missing columns: {_missing}"

print(f"Dataset loaded: {len(dataset)} items, {len(dataset.columns)} columns")
print(f"  Batches: {dict(dataset['batch'].value_counts().sort_index())}")
print(f"  Mechanisms: {len(dataset['mechanism_primary'].unique())}")

# Build answer key (item_id → spec dict)
ANSWER_KEY = {
    row['item_id']: {
        'gold_answer': row['gold_answer'],
        'aliases': row['aliases'],
        'rule': row['rule'],
    }
    for _, row in dataset.iterrows()
}

print(f"Answer key entries: {len(ANSWER_KEY)}")
assert len(ANSWER_KEY) == 102, f"Expected 102 answer key entries, got {len(ANSWER_KEY)}"




In [ ]:
# Cell 4 — Scoring & Adjudication Functions (with canonicalization pipeline)

import re
import math

# ══════════════════════════════════════════════════════════════
# Normalization (unchanged)
# ══════════════════════════════════════════════════════════════

def normalize_text(x):
    """Normalize text for answer comparison."""
    if x is None:
        return None
    return " ".join(str(x).strip().lower().split())


# ══════════════════════════════════════════════════════════════
# Wrapper stripping
# ══════════════════════════════════════════════════════════════

_WRAPPER_PREFIXES = [
    "the correct answer is",
    "the answer is",
    "the claim is",
    "the statement is",
    "the country is",
    "this is",
    "answer:",
    "answer is",
    "it is approximately",
    "it's approximately",
    "approximately",
    "it is about",
    "it's about",
    "it is",
    "it's",
    "about",
]

_HEDGE_PREFIXES = [
    "probably",
    "i think",
    "i believe",
    "maybe",
    "perhaps",
]


def strip_wrapper(text):
    """Strip light sentence wrappers from a normalized answer.

    Returns (stripped_text, was_stripped, is_hedged).
    Conservative: only strips known prefixes, trailing punctuation.
    Does NOT do fuzzy semantic matching.
    """
    if text is None:
        return None, False, False

    s = text.strip()

    # Check for hedging — return original unchanged and signal hedge
    for hedge in _HEDGE_PREFIXES:
        if s.startswith(hedge + " ") or s == hedge:
            return text, False, True

    # Strip known prefixes
    stripped = False
    for prefix in _WRAPPER_PREFIXES:
        if s.startswith(prefix + " "):
            s = s[len(prefix):].strip()
            stripped = True
            break
        elif prefix.endswith(":") and s.startswith(prefix):
            s = s[len(prefix):].strip()
            stripped = True
            break

    # Strip trailing punctuation
    s = s.rstrip(".,;:!?")

    return s.strip(), stripped, False


# ══════════════════════════════════════════════════════════════
# Numeric canonicalization
# ══════════════════════════════════════════════════════════════

_WORD_TO_DIGIT = {
    "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
    "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9",
    "ten": "10", "eleven": "11", "twelve": "12", "thirteen": "13",
    "fourteen": "14", "fifteen": "15", "sixteen": "16", "seventeen": "17",
    "eighteen": "18", "nineteen": "19", "twenty": "20",
    "thirty": "30", "forty": "40", "fifty": "50", "sixty": "60",
    "seventy": "70", "eighty": "80", "ninety": "90", "hundred": "100",
}

_COMPOUND_WORD_RE = re.compile(
    r'^(twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety)[\s-]'
    r'(one|two|three|four|five|six|seven|eight|nine)$'
)


def _word_to_number(text):
    """Convert simple number words to digit strings. Returns None if not a number word."""
    t = text.strip().lower()
    if t in _WORD_TO_DIGIT:
        return _WORD_TO_DIGIT[t]
    m = _COMPOUND_WORD_RE.match(t)
    if m:
        tens = int(_WORD_TO_DIGIT[m.group(1)])
        ones = int(_WORD_TO_DIGIT[m.group(2)])
        return str(tens + ones)
    return None


def _parse_number(text):
    """Try to parse a number from text, handling commas and scientific notation.

    Returns float or None.
    """
    if text is None:
        return None
    s = text.strip()

    # Strip commas: "299,792,458" -> "299792458"
    s_no_commas = s.replace(",", "")
    try:
        return float(s_no_commas)
    except (ValueError, TypeError):
        pass

    # Unicode scientific notation: "6.02214076 × 10²³"
    superscript_map = str.maketrans('⁰¹²³⁴⁵⁶⁷⁸⁹⁻', '0123456789-')
    sci_pattern = re.compile(
        r'([+-]?\d+\.?\d*)\s*[×x]\s*10\s*\^?\s*([⁻\-]?\s*[⁰¹²³⁴⁵⁶⁷⁸⁹0-9]+)'
    )
    m = sci_pattern.search(s)
    if m:
        base = float(m.group(1))
        exp_str = m.group(2).translate(superscript_map).replace(" ", "")
        try:
            exp = int(exp_str)
            return base * (10 ** exp)
        except ValueError:
            pass

    # Standard e-notation
    try:
        return float(s)
    except (ValueError, TypeError):
        pass

    return None


def canonicalize_numeric(text, gold_answer):
    """Extract a canonical numeric value from text for comparison with gold.

    Returns True if the extracted number matches gold, False otherwise.
    """
    gold_num = _parse_number(gold_answer)
    if gold_num is None:
        return False

    def _nums_match(a, b):
        if b == 0:
            return a == 0
        return math.isclose(a, b, rel_tol=1e-9)

    # Direct parse of the full text
    ans_num = _parse_number(text)
    if ans_num is not None:
        return _nums_match(ans_num, gold_num)

    # Number word conversion
    word_num = _word_to_number(text)
    if word_num is not None:
        ans_num = _parse_number(word_num)
        if ans_num is not None:
            return _nums_match(ans_num, gold_num)

    # Extract the single salient number from text
    numbers_found = re.findall(r'[-+]?\d[\d,]*\.?\d*(?:[eE][+-]?\d+)?', text)
    if len(numbers_found) == 1:
        ans_num = _parse_number(numbers_found[0])
        if ans_num is not None:
            return _nums_match(ans_num, gold_num)

    # Check for number words in the text tokens
    for word in text.split():
        wn = _word_to_number(word)
        if wn is not None:
            ans_num = _parse_number(wn)
            if ans_num is not None and _nums_match(ans_num, gold_num):
                return True

    return False


# ══════════════════════════════════════════════════════════════
# Yes/No/Contested canonicalization
# ══════════════════════════════════════════════════════════════

def canonicalize_yes_no(text, spec):
    """Handle yes/no/contested items after wrapper stripping.

    Also checks if the text ends with a valid yes/no/contested word
    (e.g., 'the claim is false' -> 'false').
    """
    yes_forms = {"yes", "y", "true", "correct"}
    no_forms = {"no", "n", "false", "incorrect"}
    contested_forms = {"contested", "debated", "disputed"}
    all_forms = yes_forms | no_forms | contested_forms

    canonical_gold = normalize_text(spec["gold_answer"])

    # Direct match
    if canonical_gold in contested_forms:
        if text in contested_forms:
            return True
    elif canonical_gold in yes_forms:
        if text in yes_forms:
            return True
    elif canonical_gold in no_forms:
        if text in no_forms:
            return True

    # Last-word extraction: "the claim is false" -> "false"
    words = text.split()
    if words:
        last_word = words[-1]
        if last_word in all_forms:
            if canonical_gold in contested_forms:
                return last_word in contested_forms
            elif canonical_gold in yes_forms:
                return last_word in yes_forms
            elif canonical_gold in no_forms:
                return last_word in no_forms

    # First-word extraction: "debated among experts" -> "debated"
    if words:
        first_word = words[0]
        if first_word in all_forms:
            if canonical_gold in contested_forms:
                return first_word in contested_forms
            elif canonical_gold in yes_forms:
                return first_word in yes_forms
            elif canonical_gold in no_forms:
                return first_word in no_forms

    return False


# ══════════════════════════════════════════════════════════════
# Master canonicalization orchestrator
# ══════════════════════════════════════════════════════════════

def canonicalize_answer(item_id, raw_answer, spec):
    """Orchestrate canonicalization pipeline.

    Steps:
      1. normalize_text (already done by caller)
      2. reject hedging and explanations
      3. strip_wrapper
      4. rule-specific canonicalization

    Returns True/False for correctness.
    """
    norm = raw_answer  # caller already normalized
    if norm is None:
        return False

    # Reject answers with explanation ("because" clause)
    if " because " in norm:
        return False

    # Strip wrapper
    stripped, was_stripped, is_hedged = strip_wrapper(norm)

    # Hedged answers are rejected entirely (no further canonicalization)
    if is_hedged:
        return False

    # Try alias match on stripped text
    if _grade_alias_match(stripped, spec):
        return True

    rule = spec.get("rule", "alias")

    if rule == "numeric":
        if canonicalize_numeric(stripped, spec["gold_answer"]):
            return True
        if was_stripped and canonicalize_numeric(norm, spec["gold_answer"]):
            return True
        return False

    if rule == "yes_no":
        if canonicalize_yes_no(stripped, spec):
            return True
        return False

    if rule == "alias":
        # For alias items whose gold is a yes/no/contested word,
        # also try yes_no canonicalization
        gold_norm = normalize_text(spec["gold_answer"])
        yn_forms = {"yes", "no", "true", "false", "contested", "debated", "disputed"}
        if gold_norm in yn_forms:
            if canonicalize_yes_no(stripped, spec):
                return True

    return False


# ══════════════════════════════════════════════════════════════
# Grading functions
# ══════════════════════════════════════════════════════════════

def _grade_alias_match(normalized, spec):
    """Check gold_answer first, then aliases (matching library behavior)."""
    if normalized == normalize_text(spec["gold_answer"]):
        return True
    for alias in spec.get("aliases", []):
        if normalized == normalize_text(alias):
            return True
    return False


def _grade_yes_no(normalized, spec):
    """Handle yes/no rule items (matching library behavior)."""
    yes_forms = {"yes", "y", "true", "correct"}
    no_forms = {"no", "n", "false", "incorrect"}
    canonical = normalize_text(spec["gold_answer"])
    canonical_is_yes = canonical in yes_forms
    canonical_is_no = canonical in no_forms
    if not (canonical_is_yes or canonical_is_no):
        return _grade_alias_match(normalized, spec)
    if canonical_is_yes:
        return normalized in yes_forms
    else:
        return normalized in no_forms


def adjudicate(item_id, raw_answer, gold_answer):
    """Deterministic correctness grading with canonicalization pipeline.

    Grading hierarchy:
      1. Exact normalized gold_answer match
      2. Exact normalized alias match
      3. Numeric equivalence (if rule == 'numeric')
      4. Yes/No normalization (if rule == 'yes_no')
      5. Canonicalization pipeline (wrapper stripping + rule-specific)
      6. Otherwise incorrect

    No LLM judge in the scoring loop.
    """
    spec = ANSWER_KEY.get(item_id)
    norm = normalize_text(raw_answer)
    if norm is None:
        return False

    # If no spec, fall back to exact match
    if spec is None:
        return norm == normalize_text(gold_answer)

    # 1 & 2. Gold answer + alias match (fast path)
    if _grade_alias_match(norm, spec):
        return True

    # 3. Numeric equivalence (original logic)
    if spec["rule"] == "numeric":
        try:
            if float(norm) == float(spec["gold_answer"]):
                return True
        except (ValueError, TypeError):
            pass

    # 4. Yes/No normalization (original logic)
    if spec["rule"] == "yes_no":
        if _grade_yes_no(norm, spec):
            return True

    # 5. Canonicalization pipeline (new)
    return canonicalize_answer(item_id, norm, spec)


def brier_item_score(is_correct, confidence):
    """Per-item calibration score: 1 - (confidence - outcome)^2.

    This is an affine transform of per-item Brier loss.
    Strictly proper: expected score is uniquely maximized when
    stated confidence equals true probability of correctness.

    Range: [0, 1]. Higher is better.
    Reference: Brier (1950); Gneiting & Raftery (2007).
    """
    y = 1.0 if is_correct else 0.0
    return 1.0 - (confidence - y) ** 2


print("Scoring functions defined: adjudicate(), brier_item_score()")
print("Canonicalization pipeline: strip_wrapper, canonicalize_numeric, canonicalize_yes_no")

# ══════════════════════════════════════════════════════════════
# Self-tests
# ══════════════════════════════════════════════════════════════

# ── Original self-tests (preserved) ──
print(f"\n--- Original self-tests ---")
assert adjudicate('gen_b_028', '274', '274') == True,  "gen_b_028 gold match"
assert adjudicate('gen_b_031', '10', '10') == True,    "gen_b_031 gold match"
assert adjudicate('gen_b_028', '999', '274') == False,  "gen_b_028 wrong answer"
assert adjudicate('gen_b_025', 'Italy', 'italy') == True, "gen_b_025 normalized gold"
assert adjudicate('gen_a3_004', 'no', 'false') == True, "gen_a3_004 yes_no"
assert adjudicate('gen_b3_014', '1', '1') == True,      "gen_b3_014 numeric"
print("Original self-tests passed ✓")

# ── 102/102 self-adjudication: gold answers must all pass ──
print(f"\n--- 102/102 self-adjudication (gold → gold) ---")
_self_adj_pass = 0
_self_adj_fail = []
for _iid, _spec in ANSWER_KEY.items():
    if adjudicate(_iid, _spec['gold_answer'], _spec['gold_answer']):
        _self_adj_pass += 1
    else:
        _self_adj_fail.append(_iid)
assert _self_adj_pass == 102 and not _self_adj_fail, \
    f"Self-adjudication failed for: {_self_adj_fail}"
print(f"  {_self_adj_pass}/102 gold answers self-adjudicate correctly ✓")

# ── Numeric canonicalization tests (gold='3', item gen_b3_024) ──
print(f"\n--- Numeric canonicalization tests ---")
assert adjudicate('gen_b3_024', '3', '3') == True,                       "numeric: bare '3'"
assert adjudicate('gen_b3_024', 'three', '3') == True,                   "numeric: 'three'"
assert adjudicate('gen_b3_024', '3 hearts', '3') == True,                "numeric: '3 hearts'"
assert adjudicate('gen_b3_024', 'about 3', '3') == True,                 "numeric: 'about 3'"
assert adjudicate('gen_b3_024', 'the answer is 3', '3') == True,         "numeric: 'the answer is 3'"
assert adjudicate('gen_b3_024', 'An octopus has three hearts', '3') == True, "numeric: verbose with word"
assert adjudicate('gen_b3_024', '2', '3') == False,                      "numeric: wrong number '2'"
assert adjudicate('gen_b3_024', 'two hearts', '3') == False,             "numeric: wrong word 'two hearts'"
assert adjudicate('gen_b3_024', 'probably 3 or 4', '3') == False,        "numeric: hedging"
assert adjudicate('gen_b3_024', 'I think it might be 3', '3') == False,  "numeric: hedging"
print("  Numeric tests passed ✓")

# ── Yes/no tests (gold='false') ──
print(f"\n--- Yes/no tests (gold='false') ---")
assert adjudicate('gen_a4_001', 'false', 'false') == True,               "yes_no: 'false'"
assert adjudicate('gen_a4_001', 'no', 'false') == True,                  "yes_no: 'no'"
assert adjudicate('gen_a4_001', 'the claim is false', 'false') == True,  "yes_no: wrapper"
assert adjudicate('gen_a4_001', 'answer: false', 'false') == True,       "yes_no: 'answer:'"
print("  Yes/no tests passed ✓")

# ── Contested tests (gold='contested') ──
print(f"\n--- Contested tests (gold='contested') ---")
assert adjudicate('gen_a4_022', 'contested', 'contested') == True,                "contested: bare"
assert adjudicate('gen_a4_022', 'debated', 'contested') == True,                  "contested: 'debated'"
assert adjudicate('gen_a4_022', 'this is contested', 'contested') == True,        "contested: wrapper"
assert adjudicate('gen_a4_022', 'it is debated among experts', 'contested') == True, "contested: verbose"
assert adjudicate('gen_a4_022', 'unclear', 'contested') == False,                 "contested: reject 'unclear'"
assert adjudicate('gen_a4_022', 'maybe', 'contested') == False,                   "contested: reject 'maybe'"
assert adjudicate('gen_a4_022', 'I am not sure', 'contested') == False,           "contested: reject hedging"
print("  Contested tests passed ✓")

# ── Alias tests (gold='france') ──
print(f"\n--- Alias tests (gold='france') ---")
assert adjudicate('gen_b2_021', 'france', 'france') == True,               "alias: bare"
assert adjudicate('gen_b2_021', 'France', 'france') == True,               "alias: capitalized"
assert adjudicate('gen_b2_021', 'the answer is france', 'france') == True,  "alias: wrapper"
assert adjudicate('gen_b2_021', 'it is france', 'france') == True,          "alias: wrapper"
assert adjudicate('gen_b2_021', 'probably france', 'france') == False,      "alias: hedging"
assert adjudicate('gen_b2_021', 'i think france', 'france') == False,       "alias: hedging"
assert adjudicate('gen_b2_021', 'france because it has the most visitors', 'france') == False, \
    "alias: explanation in answer"
print("  Alias tests passed ✓")

# ── Real sweep failure cases ──
print(f"\n--- Real sweep failure cases ---")
assert adjudicate('gen_a3_032', '299,792,458 meters per second', '299792458') == True, \
    "sweep: speed of light with commas and units"
assert adjudicate('gen_b3_024', 'An octopus has three hearts', '3') == True, \
    "sweep: octopus verbose"
assert adjudicate('gen_a4_022', 'this is contested', 'contested') == True, \
    "sweep: contested wrapper"
assert adjudicate('gen_b2_018', 'The answer is Canada', 'canada') == True, \
    "sweep: alias with wrapper"
assert adjudicate('gen_a3_038', '384,400 kilometers', '384400') == True, \
    "sweep: distance with commas and units"
assert adjudicate('gen_a3_035', 'The exact defined value is 6.02214076 × 10²³ mol⁻¹', '6.02214076e23') == True, \
    "sweep: Avogadro verbose with Unicode superscript"
assert adjudicate('gen_a3_037', '6.62607015 × 10⁻³⁴', '6.62607015e-34') == True, \
    "sweep: Planck constant Unicode"
print("  Real sweep failure cases passed ✓")

# ── Additional edge cases ──
print(f"\n--- Additional edge cases ---")
assert adjudicate('gen_b_040', 'contested', 'contested') == True, \
    "alias-rule contested: bare"
assert adjudicate('gen_b_040', 'this is contested', 'contested') == True, \
    "alias-rule contested: wrapper"
assert adjudicate('gen_a3_001', 'contested', 'contested') == True, \
    "yes_no contested: bare"
assert adjudicate('gen_a3_001', 'it is contested', 'contested') == True, \
    "yes_no contested: wrapper"
assert adjudicate('gen_a3_001', 'debated', 'contested') == True, \
    "yes_no contested: 'debated'"
assert adjudicate('gen_b3_031', '-4', '-4') == True, "numeric: negative"
assert adjudicate('gen_b2_034', '34,000', '34000') == True, "numeric: comma format"
print("  Additional edge cases passed ✓")

print(f"\n  brier_item_score(True, 0.9) -> {brier_item_score(True, 0.9):.4f}")
print(f"  brier_item_score(False, 0.9) -> {brier_item_score(False, 0.9):.4f}")
print(f"  brier_item_score(False, 0.2) -> {brier_item_score(False, 0.2):.4f}")
print(f"\nAll self-tests passed ✓")


In [ ]:
# Cell 5 — Benchmark Task Definition

@kbench.task(name="metacog_calibration_v1")
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str, mechanism_primary: str) -> float:
    """MetaJudge Confidence Calibration — single item evaluation.

    Workflow per item:
      1. Prompt the LLM with the calibration question
      2. Extract structured response (answer + confidence)
      3. Adjudicate correctness deterministically
      4. Compute per-item Brier-derived score
      5. Assert confidence is valid

    Returns the per-item calibration score (higher is better).
    """
    # Isolated chat so items don't leak context to each other
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{question}\n\n"
            "Instructions:\n"
            "1. Put only your final answer in the `answer` field.\n"
            "2. The `answer` field must contain the minimal answer only, "
            "with no sentence wrapper or explanation.\n"
            "3. If the question requests a number only, return only the number.\n"
            "4. If the question requests one word only, return only that word.\n"
            "5. Provide a confidence score from 0.0 to 1.0.\n"
            "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
            "7. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )

        response = llm.prompt(calibration_prompt, schema=CalibrationResponse)

    # Validate confidence range
    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    # Adjudicate correctness deterministically
    is_correct = adjudicate(item_id, response.answer, gold_answer)

    # Compute Brier-derived score
    score = brier_item_score(is_correct, conf)

    print(f"  [{item_id}] answer={response.answer!r}, "
          f"conf={conf:.2f}, correct={is_correct}, score={score:.4f}")

    return score


# Quick single-item smoke test (pulls from embedded dataset)
print("=== Smoke test (single item) ===")
_smoke = dataset.iloc[0]
result = metacog_calibration.run(
    llm=kbench.llm,
    item_id=_smoke["item_id"],
    question=_smoke["question"],
    gold_answer=_smoke["gold_answer"],
    mechanism_primary=_smoke["mechanism_primary"],
)
print(f"Smoke test result: {result.result}")


In [ ]:
# Cell 6 — Batch Evaluation (full 102-item V4.1 calibration set)

@kbench.task(name="metacog_calibration_v1_batch")
def metacog_calibration_batch(llm, df) -> float:
    """Run the full 102-item V4.1 calibration benchmark across all items.

    Returns the headline score: mean of per-item Brier-derived scores.
    This is the official MetaJudge calibration metric.
    """
    # Only pass columns that match the task function signature.
    # The SDK maps DataFrame columns → function kwargs, so extra columns
    # (aliases, rule, batch, final_tier, final_classification) cause TypeError.
    eval_cols = ["item_id", "question", "gold_answer", "mechanism_primary"]
    eval_df_input = df[eval_cols].copy()

    with kbench.client.enable_cache():
        runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == eval_df_input.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=eval_df_input,
            n_jobs=3,
        )

    eval_df = runs.as_dataframe()
    scores = eval_df["result"].astype(float)

    # Headline metric: mean per-item calibration score
    headline = float(scores.mean())

    # Diagnostics
    n_items = len(scores)
    min_score = float(scores.min())
    max_score = float(scores.max())

    print(f"\n{'='*50}")
    print(f"MetaJudge Calibration Results")
    print(f"{'='*50}")
    print(f"Items evaluated: {n_items}")
    print(f"Headline score (mean 1-Brier): {headline:.4f}")
    print(f"Score range: [{min_score:.4f}, {max_score:.4f}]")
    print(f"{'='*50}")

    return headline

# Run the full benchmark
headline_score = metacog_calibration_batch.run(kbench.llm, dataset)
print(f"\nFinal headline score: {headline_score.result}")




In [ ]:
# Cell 7 — Multi-Model Headline Sweep with Export
#
# Uses the Kaggle Benchmarks SDK evaluate() API with multiple models.
# Runs ONE MODEL AT A TIME with retries to prevent transient API
# failures from killing the entire sweep.
#
# After the sweep, logs each model run via metajudge.eval.logging
# and exports a headline CSV to outputs/sweeps/.

from metajudge.eval.logging import log_run

SWEEP_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4@20250514",
    "anthropic/claude-haiku-4-5@20251001",
    "deepseek-ai/deepseek-v3.1",
]

# Step 1: Verify model availability
print("=== Model Availability ===")
verified_models = {}
for model_name in SWEEP_MODELS:
    try:
        m = kbench.llms[model_name]
        verified_models[model_name] = m
        print(f"  \u2713 {model_name}")
    except KeyError:
        print(f"  \u2717 {model_name} \u2014 not found in kbench.llms")

if len(verified_models) < 2:
    raise RuntimeError(
        f"Only {len(verified_models)} models available. "
        f"Need \u22652 for a meaningful sweep. "
        f"Update SWEEP_MODELS with keys from: list(kbench.llms.keys())"
    )

print(f"\n{len(verified_models)}/{len(SWEEP_MODELS)} models verified.\n")

# Step 2: Run evaluate() ONE MODEL AT A TIME with retries
eval_cols = ["item_id", "question", "gold_answer", "mechanism_primary"]
eval_df_sweep = dataset[eval_cols].copy()

all_sweep_dfs = []
headline_results = {}

for model_name, model_obj in verified_models.items():
    print(f"\n{'='*60}")
    print(f"  SWEEPING: {model_name}")
    print(f"{'='*60}")

    max_retries = 3
    for attempt in range(1, max_retries + 1):
        try:
            with kbench.client.enable_cache():
                model_runs = metacog_calibration.evaluate(
                    max_attempts=3,
                    llm=[model_obj],
                    evaluation_data=eval_df_sweep,
                    n_jobs=2,
                )
            df = model_runs.as_dataframe()
            df["model"] = model_name
            all_sweep_dfs.append(df)

            # Compute headline stats
            scores = df["result"].astype(float)
            headline = float(scores.mean())
            headline_results[model_name] = {
                "headline_score": headline,
                "n_items": len(df),
            }

            print(f"\n  \u2713 {model_name}: {len(df)} rows, headline={headline:.4f}")
            break
        except Exception as e:
            print(f"\n  \u26a0 Attempt {attempt}/{max_retries} failed: {e}")
            if attempt < max_retries:
                import time
                wait = 30 * attempt
                print(f"    Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  \u2717 {model_name}: all retries exhausted, skipping")

# Step 3: Combine results and export
import pandas as pd
from pathlib import Path

if all_sweep_dfs:
    sweep_df = pd.concat(all_sweep_dfs, ignore_index=True)

    # Log each model run
    for model_name, stats in headline_results.items():
        log_run(
            run_type="headline_sweep",
            model_name=model_name,
            n_items=stats["n_items"],
            headline_score=stats["headline_score"],
            accuracy=0.0,  # not available from evaluate() results
            mean_confidence=0.0,
        )

    # Export headline CSV
    sweep_dir = Path("outputs/sweeps")
    sweep_dir.mkdir(parents=True, exist_ok=True)
    from datetime import datetime, timezone
    ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    csv_path = sweep_dir / f"headline_sweep_{ts}.csv"
    sweep_df.to_csv(csv_path, index=False)

    # Print leaderboard
    print(f"\n{'='*60}")
    print(f"SWEEP COMPLETE")
    print(f"{'='*60}")
    print(f"Models completed: {len(all_sweep_dfs)}/{len(verified_models)}")
    print(f"Total rows: {len(sweep_df)}")
    print(f"\nHeadline Leaderboard:")
    for name in sorted(headline_results, key=lambda n: -headline_results[n]["headline_score"]):
        s = headline_results[name]
        print(f"  {name:<45} {s['headline_score']:.4f}  ({s['n_items']} items)")
    print(f"\nExported: {csv_path}")
else:
    print("\nNo models completed successfully.")


In [ ]:
# Cell 8 — Detailed Audit Sweep with Per-Item Export
#
# Runs each model sequentially through all 102 items, capturing
# the FULL per-item audit trail: model_answer, confidence,
# is_correct, brier_score, would_verify.
#
# After the sweep, converts to a standard audit DataFrame via
# metajudge.eval.export and writes all artifacts via
# metajudge.eval.diagnostics.export_artifacts().
#
# Set RUN_AUDIT = True to execute. Default is False to avoid
# accidental quota burn.

RUN_AUDIT = False  # <- Set to True to run the full audit sweep

if not RUN_AUDIT:
    print("Audit sweep disabled. Set RUN_AUDIT = True in this cell to run.")
    print("This will make ~510 LLM calls and take ~10 minutes.")
    print("Use Cell 7 for quick multi-model headline scores instead.")
else:
    from metajudge.eval.export import sweep_results_to_audit_df
    from metajudge.eval.diagnostics import export_artifacts

    # SWEEP_MODELS fallback (in case Cell 7 was not run)
    if 'SWEEP_MODELS' not in dir():
        SWEEP_MODELS = [
            "google/gemini-2.5-flash",
            "google/gemini-2.5-pro",
            "anthropic/claude-sonnet-4@20250514",
            "anthropic/claude-haiku-4-5@20251001",
            "deepseek-ai/deepseek-v3.1",
        ]

    # Use same SWEEP_MODELS from Cell 7
    audit_models = {}
    for model_name in SWEEP_MODELS:
        try:
            audit_models[model_name] = kbench.llms[model_name]
        except KeyError:
            pass

    print(f"=== Audit Sweep: {len(audit_models)} models x {len(dataset)} items ===\n")

    sweep_results = {}  # model_name -> [per-item dicts]

    with kbench.client.enable_cache():
        for model_name, model in audit_models.items():
            print(f"{'='*60}")
            print(f"  MODEL: {model_name}")
            print(f"{'='*60}")

            model_items = []
            for _, row in dataset.iterrows():
                try:
                    with chats.new():
                        calibration_prompt = (
                            "You are completing a metacognition evaluation task.\n\n"
                            "Task: Confidence Calibration\n"
                            f"Question:\n{row['question']}\n\n"
                            "Instructions:\n"
                            "1. Put only your final answer in the `answer` field.\n"
                            "2. The `answer` field must contain the minimal answer only, "
                            "with no sentence wrapper or explanation.\n"
                            "3. If the question requests a number only, return only the number.\n"
                            "4. If the question requests one word only, return only that word.\n"
                            "5. Provide a confidence score from 0.0 to 1.0.\n"
                            "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
                            "7. Say whether you would verify this if possible.\n\n"
                            "Return valid structured output with keys: "
                            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
                        )
                        resp = model.prompt(calibration_prompt, schema=CalibrationResponse)

                    conf = max(0.0, min(1.0, resp.confidence))
                    correct = adjudicate(row['item_id'], resp.answer, row['gold_answer'])
                    score = brier_item_score(correct, conf)

                    model_items.append({
                        "item_id": row['item_id'],
                        "mechanism_primary": row['mechanism_primary'],
                        "gold_answer": row['gold_answer'],
                        "model_answer": str(resp.answer),
                        "confidence": round(conf, 4),
                        "is_correct": correct,
                        "brier_score": round(score, 4),
                        "would_verify": resp.would_verify_if_possible,
                    })

                    mark = "\u2713" if correct else "\u2717"
                    print(f"  [{row['item_id']}] {mark} conf={conf:.2f} "
                          f"score={score:.4f} -> {resp.answer!r}")

                except Exception as e:
                    print(f"  [{row['item_id']}] ERROR: {e}")
                    model_items.append({
                        "item_id": row['item_id'],
                        "mechanism_primary": row['mechanism_primary'],
                        "gold_answer": row['gold_answer'],
                        "model_answer": f"ERROR: {e}",
                        "confidence": 0.0,
                        "is_correct": False,
                        "brier_score": 0.0,
                        "would_verify": None,
                    })

            sweep_results[model_name] = model_items

            # Per-model summary
            scores = [r["brier_score"] for r in model_items]
            correct_count = sum(1 for r in model_items if r["is_correct"])
            headline = sum(scores) / len(scores) if scores else 0.0
            print(f"\n  -> {correct_count}/{len(dataset)} correct, headline={headline:.4f}\n")

    # Convert to standard audit DataFrame and export all artifacts
    audit_df = sweep_results_to_audit_df(sweep_results)
    artifact_paths = export_artifacts(audit_df)

    print(f"\n{'='*60}")
    print(f"AUDIT SWEEP COMPLETE")
    print(f"{'='*60}")
    print(f"Models: {len(sweep_results)}")
    print(f"Total rows: {len(audit_df)}")
    print(f"\nExported artifacts:")
    for name, path in artifact_paths.items():
        print(f"  {name}: {path}")
    print(f"\nRun Cell 9 for diagnostics.")


In [ ]:
# Cell 9 — Cross-Model Diagnostics & Success Criteria
#
# Works two ways:
#   1. After Cell 8: uses in-memory audit_df
#   2. Standalone: loads most recent audit CSV from outputs/audit/
#
# Computes model summaries, discrimination map, and C1-C5 verdict
# using the metajudge.eval.diagnostics package.

from metajudge.eval.diagnostics import (
    compute_model_summary,
    compute_discrimination,
    compute_success_criteria,
)
from pathlib import Path
import pandas as pd

# Try in-memory first, then load from file
if 'audit_df' not in dir() or audit_df is None or len(audit_df) == 0:
    audit_dir = Path("outputs/audit")
    audit_files = sorted(audit_dir.glob("per_item_audit_*.csv"))
    if audit_files:
        audit_df = pd.read_csv(audit_files[-1])
        print(f"Loaded audit data from: {audit_files[-1]}")
    else:
        print("No audit data found. Run Cell 8 with RUN_AUDIT=True first.")
        audit_df = None

if audit_df is not None and len(audit_df) > 0:
    # 1. Model Summary
    summary = compute_model_summary(audit_df)
    print("="*70)
    print("MODEL SUMMARY")
    print("="*70)
    print(summary.to_string(index=False))

    # 2. Discrimination Map
    disc = compute_discrimination(audit_df)
    print(f"\n{'='*70}")
    print(f"DISCRIMINATION MAP ({len(disc)} items where models disagree)")
    print("="*70)
    if len(disc) > 0:
        for _, row in disc.head(20).iterrows():
            print(f"  {row['item_id']}: {row['n_correct']}/{row['n_models']} correct  "
                  f"correct=[{row['models_correct']}]  wrong=[{row['models_wrong']}]")
        if len(disc) > 20:
            print(f"  ... and {len(disc) - 20} more")
    else:
        print("  No disagreements found (all models agree on all items)")

    # 3. Success Criteria Verdict
    verdict = compute_success_criteria(audit_df)
    print(f"\n{'='*70}")
    print("SUCCESS CRITERIA VERDICT")
    print("="*70)
    for key, crit in verdict["criteria"].items():
        status = "\u2713" if crit["pass"] else "\u2717"
        print(f"  [{status}] {key}: {crit['name']} "
              f"(threshold: {crit['threshold']}, measured: {crit['measured']})")

    print(f"\n  VERDICT: {verdict['total_pass']}/{verdict['total_criteria']} -> {verdict['verdict']}")
    print(f"  Models: {verdict['n_models']} ({', '.join(str(m) for m in verdict['models'])})")


In [ ]:
%choose metacog_calibration_v1_batch